# Language to Motion — CBD as Learning Data

**Demo B of the [Common Behavior Data](https://github.com/Koichi3333/common-behavior-data) project.**

Demo A turned a video into behavior data. This notebook runs the loop in the
other direction: it treats that behavior data as a **training corpus** and
learns to generate CBD-compatible behavior from a natural-language
instruction — which then replays through the very same adapters.

> ⚠️ **Read this first.** At the current scale (a handful of episodes) this is
> a **small VLA-like learning prototype**, not a general-purpose VLA. What it
> demonstrates is that the loop closes end to end — memorisation and
> interpolation, plus language-conditioned selection between learned
> behaviors. It does **not** generalise to unseen instructions. More detail in
> "Honest expectations" below.

## Why no annotation step is needed

Every line of Demo A's `frames.jsonl` already contains both halves of a
supervised example, because they were written onto the same timeline:

| Role | Field |
|---|---|
| Vision (V) | `frame_image` — the source frame |
| Language (L) | `caption` — temporal caption |
| Action (A) | `bone_rotations_xyzw` / `hips_position` / `finger_curls_rad` |
| State | `phase` — Idle / Reach / Grasp / Carry / Release |

The prompt and the answer are already paired. That is what makes a behavior
representation worth having.

## Architecture

```text
frame image (timeline/frames/*.jpg)     "The person picks up..."  (instruction)
        │ frozen ResNet18                        │ char-level CNN encoder
        ▼                                        ▼
  [vision prefix]                          [text prefix]
        └──────────────────┬──────────────────────┘
                           ▼   modality dropout during training
                               → text-only / image-only / both, one model
              causal Transformer (4 layers, d=256)
              ┌────────────┴────────────┐
              ▼                         ▼
        motion head              phase head  ← the state machine
    quat×19 + hips + curls    Idle/Reach/Grasp/Carry/Release/<end>
              │                         │ predicted phase feeds the next step
              ▼                         ▼
        CBD-compatible generated behavior
              │
   ┌──────────┼──────────────┐
   ▼          ▼              ▼
frames.jsonl  motion.vrma    humanoid.xml + motion.npz
 (data)       (Unity)        (MuJoCo)
```

Three design points worth noticing:

- **Three layers of state machine.** (1) Demo A's rule-based phase detection
  produces the supervision signal, (2) the phase head learns probabilistic
  transitions, (3) generation applies a hard grammar allowing only transitions
  actually observed in the data. A cell at the end compares the data's
  transition matrix with the generated one, so you can see how much of the
  state machine was actually learned.
- **Differentiable throughout.** Quaternions get a normalisation layer and a
  geodesic loss; positions and curls use MSE; phase uses cross-entropy;
  velocity and acceleration losses buy temporal smoothness.
- **One schema, one set of adapters.** Replaying observed motion and replaying
  generated motion use the exact same `motion.vrma` and `humanoid.xml` +
  `motion.npz` path.

## Preparing the corpus

```text
behavior_corpus/          ← CORPUS_DIR from [1]
├── Episode1/             ← one Demo A output bundle (folder name is free)
│   └── ... 04_behavior_dataset/timeline/frames.jsonl
├── Episode2/             ← just unzip demo2_output_bundle.zip here
└── EpisodeN/             ← add more and re-run to train on more
```

`timeline/frames.jsonl` is found recursively, so the exact nesting does not
matter. **Episodes with captions (Demo A cell `[5.5]`) and frame images
(`video.export_frame_images = True`, on by default) are recommended.** Mixing
in episodes that lack one of them works too — the missing modality becomes a
learned null embedding.

## Runtime

T4 / L4 / A100 all work, and so does CPU (just slower). The model is a few
million parameters, sized for small corpora, and uses AMP (fp16) automatically.

## Honest expectations

Training on a few episodes demonstrates **that the mechanism runs end to
end**; in substance it is memorisation and interpolation of the training data.
The closer a prompt is to a learned caption, the better the output.

**What works today**

- Faithful regeneration of a learned behavior from a prompt close to its caption
- With 2–3 episodes of *different* behaviors, selecting between them by instruction
- Grammatically valid phase sequences (Idle → Reach → Grasp → Carry → Release → end)
- Conditioning on a frame image from the training episodes

**What does not**

- Unseen instructions ("with your left hand", "wave both arms")
- Non-English prompts (training captions are English)
- Object trajectory generation (current output is the human body only)
- Physically consistent contact or grasping

Realistic estimate: early signs of interpolation appear around 10 episodes;
anything resembling generalisation needs roughly 100 episodes of the same task
family. The schema, state machine and loss design are the same shape as a real
VLA, so the structure is built to keep improving as episodes accumulate.


In [ ]:
# =====================================================================
# [1] Environment setup
#     Pure PyTorch, small model, automatic device selection.
#     Runs on T4 / L4 / A100 -- and on CPU, just slower.
# =====================================================================
import subprocess, sys

for package in ["mujoco"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=True)

import json
import math
import os
import random
import shutil
import struct
from collections import Counter
from pathlib import Path

import numpy as np

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch"], check=True)
    import torch
    import torch.nn as nn
    import torch.nn.functional as F

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU"
USE_AMP = DEVICE == "cuda"   # fp16 autocast works on T4 / L4 / A100 alike
print(f"device={DEVICE} ({GPU_NAME})  AMP={USE_AMP}  torch={torch.__version__}")

# ------------------------- Where the corpus lives -------------------------
# One directory per episode, where an episode is one Demo A output bundle.
# All an episode really needs is 04_behavior_dataset/timeline/frames.jsonl --
# unzipping demo2_output_bundle.zip in place is enough.
#
#   behavior_corpus/
#   |-- episode_001/ ... 04_behavior_dataset/ (contains frames.jsonl)
#   |-- episode_002/ ...
#   `-- episode_XXX/ ...
USE_GOOGLE_DRIVE = False
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    CORPUS_DIR = Path("/content/drive/MyDrive/behavior_corpus")
else:
    CORPUS_DIR = Path("/content/behavior_corpus")
CORPUS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT = CORPUS_DIR / "_vla_runs"
OUTPUT_ROOT.mkdir(exist_ok=True)
print("CORPUS_DIR:", CORPUS_DIR)


In [ ]:
# =====================================================================
# [1.5] Prepare a drop zone for corpus ZIPs
#     Creates /content/upload_zips. Drag your demo2_output_bundle.zip files
#     there in the Colab file pane (as many as you have), then run [1.6].
#     More episodes -> better generation; see the honest expectations below.
# =====================================================================

import os
import zipfile

# =========================
# Folders
# =========================
ZIP_DIR = "/content/upload_zips"        # drop uploaded ZIPs here
OUTPUT_DIR = "/content/behavior_corpus" # episodes are extracted here

os.makedirs(ZIP_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# =====================================================================
# [1.6] Extract each ZIP into its own episode folder
#     Every ZIP in upload_zips becomes behavior_corpus/EpisodeN.
#     A ZIP is deleted once extracted, and kept on failure.
#     No need to tidy the folder layout: [2] finds frames.jsonl recursively.
# =====================================================================
zip_files = sorted([
    f for f in os.listdir(ZIP_DIR)
    if f.lower().endswith(".zip")
])

if not zip_files:
    print(f"No ZIP files found in {ZIP_DIR}")
else:
    print(f"Processing {len(zip_files)} ZIP file(s).\n")

    for i, zip_name in enumerate(zip_files, start=1):
        zip_path = os.path.join(ZIP_DIR, zip_name)

        # Episode1, Episode2, ...
        episode_dir = os.path.join(OUTPUT_DIR, f"Episode{i}")
        os.makedirs(episode_dir, exist_ok=True)

        print(f"[{i}/{len(zip_files)}] extracting: {zip_name}")
        print(f"  -> {episode_dir}")

        try:
            with zipfile.ZipFile(zip_path, "r") as zf:
                zf.extractall(episode_dir)

            # Only delete the ZIP once extraction actually succeeded
            os.remove(zip_path)

            print("  [ok] extracted, ZIP removed")

        except Exception as e:
            print(f"  [fail] {e}")
            print("  -> keeping the ZIP so you can retry.")

print("\nDone")
print(f"Extracted into: {OUTPUT_DIR}")

In [ ]:
# =====================================================================
# [2] Load the corpus: frames.jsonl -> (text, motion, phase sequence)
#     No annotation step is needed here. Each line of frames.jsonl already
#     pairs the prompt (caption + frame image) with the answer (bone
#     rotations, hips, finger curls, phase), because Demo A wrote them onto
#     the same timeline. That alignment is the whole point of the format.
# =====================================================================
BONE_ORDER = [
    "hips", "spine", "chest", "neck", "head",
    "left_shoulder", "left_upper_arm", "left_lower_arm", "left_hand",
    "right_shoulder", "right_upper_arm", "right_lower_arm", "right_hand",
    "left_upper_leg", "left_lower_leg", "left_foot",
    "right_upper_leg", "right_lower_leg", "right_foot",
]
MOTION_DIM = len(BONE_ORDER) * 4 + 3 + 10   # quat(xyzw)x19 + hips(3) + curls(10)

# ---- Lower-body freeze (a deliberate, temporary measure) ----------------
# When True the six leg bones are pinned to a fixed pose and excluded from
# both training and generation.
#
# Why it is needed right now: the current corpus is a handful of seated
# videos, and two effects compound:
#   1) seated legs are hidden behind the table, so MediaPipe's estimate there
#      is low confidence,
#   2) with few episodes there is no averaging to cancel that noise out.
# The model then faithfully learns the noise -- which is exactly why the legs
# jitter in generated motion.
#
# So this is not an accuracy problem, it is a training-data problem: freezing
# the legs frees the model's capacity for the upper body that IS observed.
#
# Turn it off when the corpus grows:
#   - always set False if you include standing / walking episodes,
#   - even for seated-only data, once you have enough episodes, set False and
#     compare quaternion loss and generation stability.
FREEZE_LOWER_BODY = True
# Which pose to freeze into:
#   "standing" ... upright, legs straight down (T-pose equivalent)
#   "sitting"  ... seated, thighs horizontal and shins vertical
# For seated source video, "sitting" simply looks right.
FREEZE_POSE = "sitting"

LOWER_BODY_BONES = ["left_upper_leg", "left_lower_leg", "left_foot",
                    "right_upper_leg", "right_lower_leg", "right_foot"]
IDENTITY_QUAT_XYZW = np.array([0.0, 0.0, 0.0, 1.0], dtype=np.float32)

# Local rotations for the seated pose (canonical frame, xyzw).
# Verified in MuJoCo to give thighs horizontal-forward and shins straight
# down: -90 deg at the hip, +90 deg at the knee, both parent-relative.
_S = float(np.sin(np.pi / 4))
SITTING_QUATS_XYZW = {
    "left_upper_leg":  np.array([-_S, 0.0, 0.0, _S], dtype=np.float32),
    "right_upper_leg": np.array([-_S, 0.0, 0.0, _S], dtype=np.float32),
    "left_lower_leg":  np.array([_S, 0.0, 0.0, _S], dtype=np.float32),
    "right_lower_leg": np.array([_S, 0.0, 0.0, _S], dtype=np.float32),
    "left_foot":       IDENTITY_QUAT_XYZW,   # ankle stays square to the shin
    "right_foot":      IDENTITY_QUAT_XYZW,
}


def freeze_lower_body(motion, pose=None):
    """Replace the leg bones with a fixed pose. Upper body and hips untouched."""
    pose = pose or FREEZE_POSE
    motion = motion.copy()
    for bone in LOWER_BODY_BONES:
        base = BONE_ORDER.index(bone) * 4
        if pose == "sitting":
            motion[:, base:base + 4] = SITTING_QUATS_XYZW[bone]
        else:
            motion[:, base:base + 4] = IDENTITY_QUAT_XYZW
    return motion
MAX_SEQ_FRAMES = 240        # longest sample, 16 s at 15 fps
FPS = 15.0

def frame_to_vector(rec):
    human = rec["human"]
    parts = []
    for bone in BONE_ORDER:
        parts.extend(human["bone_rotations_xyzw"][bone])
    parts.extend(human["hips_position"])
    for side in ["left", "right"]:
        curls = human["finger_curls_rad"].get(side)
        parts.extend(curls if curls else [0.0] * 5)
    return np.array(parts, dtype=np.float32)


def load_episode(frames_jsonl):
    records = [json.loads(line) for line in
               frames_jsonl.read_text().splitlines() if line.strip()]
    motion = np.stack([frame_to_vector(r) for r in records])
    if FREEZE_LOWER_BODY:
        motion = freeze_lower_body(motion)
    phases = [r["phase"]["phase"] for r in records]
    captions = [(r.get("caption") or {}).get("en") for r in records]
    # Frame images written by Demo A into timeline/frames/ (None if absent)
    dataset_root = frames_jsonl.parent.parent
    images = []
    for r in records:
        rel = r.get("frame_image")
        path = dataset_root / rel if rel else None
        images.append(str(path) if path and path.exists() else None)
    # behavior_summary description, used as the whole-episode caption
    summary_path = frames_jsonl.parent.parent / "behavior_summary.json"
    summary_text = None
    if summary_path.exists():
        summary = json.loads(summary_path.read_text())
        summary_text = summary.get("description")
    return {"motion": motion, "phases": phases, "captions": captions,
            "images": images, "summary": summary_text,
            "path": str(frames_jsonl)}


EPISODES = []
for frames_jsonl in sorted(CORPUS_DIR.rglob("timeline/frames.jsonl")):
    if "_vla_runs" in str(frames_jsonl):
        continue
    try:
        EPISODES.append(load_episode(frames_jsonl))
    except Exception as error:
        print(f"  failed to load, skipping: {frames_jsonl} ({error})")
if not EPISODES:
    raise RuntimeError(
        f"No episodes found. Put Demo A output folders (each containing "
        f"04_behavior_dataset/) under {CORPUS_DIR}.")

# ---- Phase vocabulary. <end> terminates a sequence; together these are the
#      state set of the behavior state machine. ----
PHASE_VOCAB = sorted({p for ep in EPISODES for p in ep["phases"]})
PHASE_VOCAB.append("<end>")
PHASE_TO_ID = {p: i for i, p in enumerate(PHASE_VOCAB)}
NUM_PHASES = len(PHASE_VOCAB)

# ---- Training samples: (text, motion segment, phase segment) ----
#   a) per caption window: that window's English caption <-> its motion
#   b) whole episode:      the summary sentence <-> the full motion (clipped)
SAMPLES = []
for ep in EPISODES:
    T = len(ep["motion"])
    # a) caption windows
    start = 0
    while start < T:
        cap = ep["captions"][start]
        end = start
        while end + 1 < T and ep["captions"][end + 1] == cap:
            end += 1
        if cap:
            mid = (start + end) // 2
            SAMPLES.append({"text": cap,
                            "motion": ep["motion"][start:end + 1],
                            "phases": ep["phases"][start:end + 1],
                            "image": ep["images"][mid]})   # middle frame = vision
        start = end + 1
    # b) the episode as a whole
    text = ep["summary"] or "a person performs a manipulation task"
    SAMPLES.append({"text": text, "motion": ep["motion"][:MAX_SEQ_FRAMES],
                    "phases": ep["phases"][:MAX_SEQ_FRAMES],
                    "image": ep["images"][min(len(ep["images"]) - 1,
                                              MAX_SEQ_FRAMES // 2)]})

# ---- Normalisation stats. Only hips and curls need it -- quaternions are
#      already unit length, so they are left untouched. ----
all_motion = np.concatenate([s["motion"] for s in SAMPLES])
STATS_MEAN = all_motion.mean(axis=0)
STATS_STD = all_motion.std(axis=0) + 1e-6
QUAT_SLICE = slice(0, len(BONE_ORDER) * 4)
STATS_MEAN[QUAT_SLICE] = 0.0
STATS_STD[QUAT_SLICE] = 1.0

def normalize(motion):
    return (motion - STATS_MEAN) / STATS_STD

def denormalize(motion):
    return motion * STATS_STD + STATS_MEAN

# ---- Phase transition matrix measured from the data. After training we
#      compare the generated transitions against this. ----
DATA_TRANSITIONS = np.zeros((NUM_PHASES, NUM_PHASES))
for ep in EPISODES:
    ids = [PHASE_TO_ID[p] for p in ep["phases"]] + [PHASE_TO_ID["<end>"]]
    for a, b in zip(ids, ids[1:]):
        DATA_TRANSITIONS[a, b] += 1
START_PHASE_IDS = sorted({PHASE_TO_ID[ep["phases"][0]] for ep in EPISODES})
row_sums = DATA_TRANSITIONS.sum(axis=1, keepdims=True)
DATA_TRANSITIONS = np.divide(DATA_TRANSITIONS, row_sums,
                             out=np.zeros_like(DATA_TRANSITIONS),
                             where=row_sums > 0)

with_image = sum(1 for s in SAMPLES if s.get("image"))
print(f"samples with a frame image: {with_image}/{len(SAMPLES)}")
if FREEZE_LOWER_BODY:
    print(f"lower body frozen: ON (pose={FREEZE_POSE}) -- temporary measure "
          "for seated, few-episode corpora; set FREEZE_LOWER_BODY=False once "
          "you have more data")
print(f"episodes={len(EPISODES)}  samples={len(SAMPLES)}  "
      f"motion_dim={MOTION_DIM}  phases={PHASE_VOCAB}")
print(f"total frames: {sum(len(e['motion']) for e in EPISODES)} "
      f"({sum(len(e['motion']) for e in EPISODES) / FPS:.1f}s)")


In [ ]:
# =====================================================================
# [3] Model: a causal Transformer from (vision, language) to motion + phase
#
#   [vision token, text token] -> causal Transformer ->
#        |-- motion head : regress the next frame (quaternions / hips / curls)
#        `-- phase head  : classify the next phase (the state-machine step)
#
#   The state machine is the interesting half. The phase head learns
#   "current state -> next state", and at generation time the predicted phase
#   is fed back in as input for the next step. Everything is differentiable;
#   quaternions go through a normalisation layer and a geodesic loss.
# =====================================================================

# ---- Text encoder: character-level CNN + mean pool.
#      No tokeniser dependency, and it degrades gracefully on unseen words. ----
class CharTextEncoder(nn.Module):
    def __init__(self, embed_dim=256, out_dim=256, max_chars=160):
        super().__init__()
        self.max_chars = max_chars
        self.char_embed = nn.Embedding(8192, embed_dim)   # unicode codepoint % 8192
        self.conv = nn.Sequential(
            nn.Conv1d(embed_dim, embed_dim, kernel_size=3, padding=1),
            nn.GELU(),
            nn.Conv1d(embed_dim, out_dim, kernel_size=3, padding=1),
            nn.GELU())

    def encode_ids(self, texts, device):
        batch = torch.zeros(len(texts), self.max_chars, dtype=torch.long)
        for i, text in enumerate(texts):
            codes = [ord(c) % 8192 for c in text[:self.max_chars]]
            batch[i, :len(codes)] = torch.tensor(codes)
        return batch.to(device)

    def forward(self, texts, device):
        ids = self.encode_ids(texts, device)          # (B, L)
        h = self.char_embed(ids).transpose(1, 2)      # (B, E, L)
        h = self.conv(h).transpose(1, 2)              # (B, L, D)
        mask = (ids > 0).float().unsqueeze(-1)
        return (h * mask).sum(1) / mask.sum(1).clamp(min=1.0)   # (B, D) mean pool


# ---- Vision encoder: frozen ResNet18, which stays stable on tiny datasets.
#      Falls back to a small CNN if torchvision weights are unavailable. ----
class VisionEncoder(nn.Module):
    def __init__(self, out_dim=256):
        super().__init__()
        self.backbone, feat_dim = None, 512
        try:
            import torchvision
            weights = torchvision.models.ResNet18_Weights.DEFAULT
            resnet = torchvision.models.resnet18(weights=weights)
            self.backbone = nn.Sequential(*list(resnet.children())[:-1])
            for param in self.backbone.parameters():
                param.requires_grad = False        # frozen: less overfitting, less compute
            self.backbone.eval()
            print("VisionEncoder: frozen ResNet18 (ImageNet pretrained)")
        except Exception as error:
            print(f"torchvision unavailable, falling back to a small CNN ({error})")
            feat_dim = 128
            self.backbone = nn.Sequential(
                nn.Conv2d(3, 32, 5, 2, 2), nn.GELU(),
                nn.Conv2d(32, 64, 3, 2, 1), nn.GELU(),
                nn.Conv2d(64, feat_dim, 3, 2, 1), nn.GELU(),
                nn.AdaptiveAvgPool2d(1))
        self.project = nn.Linear(feat_dim, out_dim)
        mean = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
        std = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)
        self.register_buffer("img_mean", mean)
        self.register_buffer("img_std", std)

    @staticmethod
    def load_image(path, size=224):
        import cv2
        img = cv2.imread(path)
        if img is None:
            return None
        img = cv2.cvtColor(cv2.resize(img, (size, size)), cv2.COLOR_BGR2RGB)
        return torch.from_numpy(img).permute(2, 0, 1).float() / 255.0

    def forward(self, image_batch):                # (B, 3, H, W), unnormalised
        x = (image_batch - self.img_mean) / self.img_std
        with torch.no_grad() if not any(
                p.requires_grad for p in self.backbone.parameters())                 else torch.enable_grad():
            feats = self.backbone(x).flatten(1)
        return self.project(feats)                 # (B, D)


class MotionTransformer(nn.Module):
    def __init__(self, motion_dim, num_phases, d_model=256, n_heads=4,
                 n_layers=4, ff_dim=512, dropout=0.1, max_len=MAX_SEQ_FRAMES + 8):
        super().__init__()
        self.text_encoder = CharTextEncoder(out_dim=d_model)
        self.vision_encoder = VisionEncoder(out_dim=d_model)
        # Learned null embeddings, used when a modality is missing
        self.null_text = nn.Parameter(torch.zeros(1, d_model))
        self.null_image = nn.Parameter(torch.zeros(1, d_model))
        self.motion_in = nn.Linear(motion_dim, d_model)
        self.phase_embed = nn.Embedding(num_phases, d_model)
        self.start_token = nn.Parameter(torch.zeros(1, 1, d_model))
        position = torch.arange(max_len).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(position * div)
        pe[:, 1::2] = torch.cos(position * div)
        self.register_buffer("pos_encoding", pe)
        layer = nn.TransformerEncoderLayer(
            d_model, n_heads, ff_dim, dropout, batch_first=True,
            activation="gelu", norm_first=True)
        self.transformer = nn.TransformerEncoder(layer, n_layers)
        self.motion_head = nn.Linear(d_model, motion_dim)
        self.phase_head = nn.Linear(d_model, num_phases)
        self.d_model = d_model

    def encode_prefix(self, texts, images, device):
        """prefix = [vision token, text token]; a missing modality becomes its
        null embedding. Together with modality dropout during training, one
        model handles text-only, image-only and both."""
        B = len(texts)
        text_vecs = torch.stack([
            self.null_text[0] if t is None else torch.zeros(self.d_model,
                                                            device=device)
            for t in texts])
        real_texts = [t for t in texts if t is not None]
        if real_texts:
            encoded = self.text_encoder(real_texts, device)
            idx = 0
            rows = []
            for t in texts:
                if t is None:
                    rows.append(self.null_text[0])
                else:
                    rows.append(encoded[idx]); idx += 1
            text_vecs = torch.stack(rows)
        image_rows = []
        real_imgs, real_pos = [], []
        for i, img in enumerate(images):
            if img is None:
                image_rows.append(self.null_image[0])
            else:
                real_imgs.append(img); real_pos.append(i)
                image_rows.append(None)
        if real_imgs:
            feats = self.vision_encoder(torch.stack(real_imgs).to(device))
            for j, i in enumerate(real_pos):
                image_rows[i] = feats[j]
        image_vecs = torch.stack(image_rows)
        return torch.stack([image_vecs, text_vecs], dim=1)   # (B,2,D)

    def forward(self, texts, images, motion, phase_ids, device):
        """Teacher forcing: input [prefix(2), start, x_0..x_{T-2}] predicts
        [x_0..x_{T-1}]."""
        B, T, _ = motion.shape
        prefix = self.encode_prefix(texts, images, device)               # (B,2,D)
        frame_tokens = self.motion_in(motion[:, :-1]) \
            + self.phase_embed(phase_ids[:, :-1])                        # (B,T-1,D)
        tokens = torch.cat([prefix,
                            self.start_token.expand(B, 1, self.d_model),
                            frame_tokens], dim=1)                        # (B,T+2,D)
        tokens = tokens + self.pos_encoding[:tokens.shape[1]]
        causal = nn.Transformer.generate_square_subsequent_mask(
            tokens.shape[1], device=device)
        hidden = self.transformer(tokens, mask=causal, is_causal=True)
        out = hidden[:, 2:]                                              # drop the prefix
        return self.motion_head(out), self.phase_head(out)               # each (B, T, .)


def quaternion_geodesic_loss(pred, target):
    """1 - |<q_pred, q_target>|: geodesic distance that absorbs quaternion
    sign ambiguity, and stays differentiable."""
    pred = pred.view(*pred.shape[:-1], -1, 4)
    target = target.view(*target.shape[:-1], -1, 4)
    pred = F.normalize(pred, dim=-1)
    dots = (pred * target).sum(-1).abs()
    return (1.0 - dots).mean()


def motion_losses(pred_motion, target_motion, pred_phase, target_phase, mask):
    """mask: (B, T) valid frames. Combines the differentiable loss terms."""
    m = mask.unsqueeze(-1)
    quat_loss = quaternion_geodesic_loss(
        (pred_motion[..., QUAT_SLICE] * m)[mask.bool()],
        (target_motion[..., QUAT_SLICE] * m)[mask.bool()])
    other_pred = pred_motion[..., QUAT_SLICE.stop:]
    other_tgt = target_motion[..., QUAT_SLICE.stop:]
    reg_loss = (((other_pred - other_tgt) ** 2) * m).sum() / m.sum() / other_pred.shape[-1]
    ce = F.cross_entropy(pred_phase.reshape(-1, NUM_PHASES),
                         target_phase.reshape(-1), reduction="none")
    phase_loss = (ce * mask.reshape(-1)).sum() / mask.sum()
    # Temporal smoothness: match the frame-to-frame delta (velocity).
    # Without this every frame is regressed independently, and the small
    # per-frame errors of autoregressive generation show up as visible jitter.
    vel_mask = (mask[:, 1:] * mask[:, :-1]).unsqueeze(-1)
    pred_vel = pred_motion[:, 1:] - pred_motion[:, :-1]
    tgt_vel = target_motion[:, 1:] - target_motion[:, :-1]
    velocity_loss = (((pred_vel - tgt_vel) ** 2) * vel_mask).sum() \
        / vel_mask.sum().clamp(min=1.0) / pred_motion.shape[-1]
    # Match acceleration (second difference) too, damping the vibration further
    if pred_motion.shape[1] > 2:
        acc_mask = (mask[:, 2:] * mask[:, 1:-1] * mask[:, :-2]).unsqueeze(-1)
        pred_acc = pred_vel[:, 1:] - pred_vel[:, :-1]
        tgt_acc = tgt_vel[:, 1:] - tgt_vel[:, :-1]
        accel_loss = (((pred_acc - tgt_acc) ** 2) * acc_mask).sum() \
            / acc_mask.sum().clamp(min=1.0) / pred_motion.shape[-1]
        velocity_loss = velocity_loss + 0.5 * accel_loss
    return quat_loss, reg_loss, phase_loss, velocity_loss


MODEL = MotionTransformer(MOTION_DIM, NUM_PHASES).to(DEVICE)
print(f"parameters: {sum(p.numel() for p in MODEL.parameters()) / 1e6:.2f}M "
      f"(comfortably small -- a T4 is plenty)")


In [ ]:
# =====================================================================
# [4] Training loop (teacher forcing / AMP / gradient clipping)
#     Be clear about the goal at this scale: with a handful of episodes we
#     deliberately overfit, to show that the text -> motion mapping can be
#     memorised and interpolated end to end. This is not generalisation.
# =====================================================================
EPOCHS = 400
BATCH_SIZE = 8
LEARNING_RATE = 3e-4
# Scheduled sampling: later in training, replace some input frames with the
# model's own predictions. Training always on ground truth but generating from
# its own output is exposure bias -- the root cause of jitter and drift. This
# lets the model taste its own errors while it can still correct for them.
SCHEDULED_SAMPLING_MAX = 0.3

MODALITY_DROP_TEXT = 0.3    # p(drop text)  -> teaches the image-only mode
MODALITY_DROP_IMAGE = 0.3   # p(drop image) -> teaches the text-only mode
IMAGE_CACHE = {}            # avoid re-reading the same JPEG every epoch

def load_image_cached(path):
    if path not in IMAGE_CACHE:
        IMAGE_CACHE[path] = VisionEncoder.load_image(path)
    return IMAGE_CACHE[path]

def build_batch(samples, train=True):
    """Padded batch: motion (B, T, D), phase (B, T), mask (B, T) + prefixes."""
    texts, images = [], []
    for s in samples:
        text, image_path = s["text"], s.get("image")
        if train:
            drop_text = random.random() < MODALITY_DROP_TEXT
            drop_image = random.random() < MODALITY_DROP_IMAGE
            if drop_text and (drop_image or image_path is None):
                drop_text = False          # never drop both modalities
            if drop_text:
                text = None
            if drop_image:
                image_path = None
        texts.append(text)
        images.append(load_image_cached(image_path) if image_path else None)
    T = max(len(s["motion"]) for s in samples) + 1     # +1 for the <end> step
    B = len(samples)
    motion = np.zeros((B, T, MOTION_DIM), np.float32)
    phase = np.full((B, T), PHASE_TO_ID["<end>"], np.int64)
    mask = np.zeros((B, T), np.float32)
    for i, s in enumerate(samples):
        L = len(s["motion"])
        motion[i, :L] = normalize(s["motion"])
        motion[i, L] = motion[i, L - 1]                # <end> holds the last pose
        phase[i, :L] = [PHASE_TO_ID[p] for p in s["phases"]]
        mask[i, :L + 1] = 1.0
    return (texts, images,
            torch.from_numpy(motion).to(DEVICE),
            torch.from_numpy(phase).to(DEVICE),
            torch.from_numpy(mask).to(DEVICE))


optimizer = torch.optim.AdamW(MODEL.parameters(), lr=LEARNING_RATE,
                              weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, EPOCHS)
scaler = torch.amp.GradScaler(enabled=USE_AMP)
history = []

MODEL.train()
for epoch in range(1, EPOCHS + 1):
    random.shuffle(SAMPLES)
    epoch_losses = []
    for i in range(0, len(SAMPLES), BATCH_SIZE):
        texts, images, motion, phase, mask = build_batch(
            SAMPLES[i:i + BATCH_SIZE])
        optimizer.zero_grad(set_to_none=True)
        ss_prob = SCHEDULED_SAMPLING_MAX * max(
            0.0, min(1.0, (epoch / EPOCHS - 0.25) / 0.5))   # ramps in from 25% of training
        with torch.amp.autocast(DEVICE, enabled=USE_AMP):
            if ss_prob > 0:
                with torch.no_grad():   # first pass: collect the model's own predictions
                    first_motion, _ = MODEL(texts, images, motion, phase, DEVICE)
                mixed = motion.clone()
                replace = (torch.rand(motion.shape[0], motion.shape[1] - 1,
                                      device=DEVICE) < ss_prob)
                # Under AMP the first pass returns fp16; match the target dtype
                mixed[:, :-1][replace] = \
                    first_motion.detach()[:, :-1][replace].to(mixed.dtype)
            else:
                mixed = motion
            pred_motion, pred_phase = MODEL(texts, images, mixed, phase, DEVICE)
            quat_l, reg_l, phase_l, vel_l = motion_losses(
                pred_motion, motion, pred_phase, phase, mask)
            loss = quat_l + reg_l + 0.5 * phase_l + 2.0 * vel_l
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(MODEL.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        epoch_losses.append([loss.item(), quat_l.item(), reg_l.item(),
                             phase_l.item(), vel_l.item()])
    scheduler.step()
    mean = np.mean(epoch_losses, axis=0)
    history.append(mean)
    if epoch % 25 == 0 or epoch == 1:
        print(f"epoch {epoch:4d}  loss={mean[0]:.4f}  "
              f"(quat={mean[1]:.4f} reg={mean[2]:.4f} phase={mean[3]:.4f} "
              f"vel={mean[4]:.4f})")

# Loss curves
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
history_arr = np.array(history)
plt.figure(figsize=(8, 4))
for idx, name in enumerate(["total", "quat", "hips/curls", "phase",
                            "velocity"]):
    plt.plot(history_arr[:, idx], label=name)
plt.legend(); plt.xlabel("epoch"); plt.ylabel("loss"); plt.yscale("log")
plt.title("training losses"); plt.tight_layout()
plt.savefig(OUTPUT_ROOT / "loss_curve.png", dpi=100)
plt.show()

CHECKPOINT_PATH = OUTPUT_ROOT / "vla_checkpoint.pt"
torch.save({"model": MODEL.state_dict(),
            "stats_mean": STATS_MEAN, "stats_std": STATS_STD,
            "phase_vocab": PHASE_VOCAB, "bone_order": BONE_ORDER},
           CHECKPOINT_PATH)
print("checkpoint:", CHECKPOINT_PATH)


In [ ]:
# =====================================================================
# [4.5] Save / load / fine-tune a checkpoint (optional cell)
#
#   Pick a MODE and run:
#     "save"     ... save the current MODEL (optionally backed up to Drive)
#     "load"     ... load a saved .pt, letting you skip training in [4]
#     "finetune" ... continue training from a saved .pt at a low LR
#
#   Important: fine-tuning on new data ALONE makes the model forget old
#   behaviors fast (catastrophic forgetting). So "finetune" here re-trains on
#   the whole current corpus -- old and new mixed -- at a low learning rate.
#   After adding a new behavior, add its ZIP in [1.6] and re-run [2] first.
# =====================================================================
MODE = "save"                       # "save" / "load" / "finetune"
CHECKPOINT_FILE = OUTPUT_ROOT / "vla_checkpoint.pt"
DRIVE_BACKUP_DIR = None             # e.g. "/content/drive/MyDrive/vla_checkpoints"
FINETUNE_EPOCHS = 120
FINETUNE_LR = 5e-5                  # well below the 3e-4 used for training


def save_checkpoint(path):
    torch.save({
        "model": MODEL.state_dict(),
        "stats_mean": STATS_MEAN, "stats_std": STATS_STD,
        "phase_vocab": PHASE_VOCAB, "bone_order": BONE_ORDER,
        "motion_dim": MOTION_DIM,
        "freeze_lower_body": FREEZE_LOWER_BODY,
        "freeze_pose": FREEZE_POSE,
        "episodes": len(EPISODES), "samples": len(SAMPLES),
    }, path)
    print(f"Saved: {path} ({Path(path).stat().st_size / 1024 / 1024:.1f} MB)")


def load_checkpoint(path, strict_check=True):
    """Load weights after checking compatibility. On a structural mismatch we
    stop and say why, rather than silently producing broken output."""
    ckpt = torch.load(path, map_location=DEVICE, weights_only=False)
    problems = []
    if ckpt.get("motion_dim") != MOTION_DIM:
        problems.append(f"motion_dim: saved={ckpt.get('motion_dim')} / current={MOTION_DIM}")
    if list(ckpt.get("bone_order", [])) != list(BONE_ORDER):
        problems.append("bone_order differs")
    if list(ckpt.get("phase_vocab", [])) != list(PHASE_VOCAB):
        problems.append(
            f"phase_vocab differs (saved={ckpt.get('phase_vocab')} / "
            f"current={PHASE_VOCAB})"
            "\n      -> this happens when you add a new behavior. The phase "
            "head no longer matches, so retrain from [4].")
    if problems and strict_check:
        raise RuntimeError("Incompatible checkpoint:\n  - "
                           + "\n  - ".join(problems))
    MODEL.load_state_dict(ckpt["model"])
    globals()["STATS_MEAN"] = ckpt["stats_mean"]
    globals()["STATS_STD"] = ckpt["stats_std"]
    print(f"Loaded: {path}")
    print(f"  trained on: {ckpt.get('episodes')} episodes / "
          f"{ckpt.get('samples')} samples")
    print(f"  lower body frozen: {ckpt.get('freeze_lower_body')} "
          f"(pose={ckpt.get('freeze_pose')})")
    return ckpt


def finetune(epochs=FINETUNE_EPOCHS, lr=FINETUNE_LR):
    """Continue training on the whole current corpus at a low learning rate."""
    optimizer_ft = torch.optim.AdamW(MODEL.parameters(), lr=lr,
                                     weight_decay=0.01)
    scheduler_ft = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_ft, epochs)
    scaler_ft = torch.amp.GradScaler(enabled=USE_AMP)
    MODEL.train()
    for epoch in range(1, epochs + 1):
        random.shuffle(SAMPLES)
        losses = []
        for i in range(0, len(SAMPLES), BATCH_SIZE):
            texts, images, motion, phase, mask = build_batch(
                SAMPLES[i:i + BATCH_SIZE])
            optimizer_ft.zero_grad(set_to_none=True)
            with torch.amp.autocast(DEVICE, enabled=USE_AMP):
                pred_motion, pred_phase = MODEL(texts, images, motion, phase,
                                                DEVICE)
                quat_l, reg_l, phase_l, vel_l = motion_losses(
                    pred_motion, motion, pred_phase, phase, mask)
                loss = quat_l + reg_l + 0.5 * phase_l + 2.0 * vel_l
            scaler_ft.scale(loss).backward()
            scaler_ft.unscale_(optimizer_ft)
            nn.utils.clip_grad_norm_(MODEL.parameters(), 1.0)
            scaler_ft.step(optimizer_ft)
            scaler_ft.update()
            losses.append([loss.item(), quat_l.item(), phase_l.item()])
        scheduler_ft.step()
        if epoch % 20 == 0 or epoch == 1:
            mean = np.mean(losses, axis=0)
            print(f"  finetune epoch {epoch:4d}  loss={mean[0]:.4f} "
                  f"(quat={mean[1]:.4f} phase={mean[2]:.4f})")


# ------------------------------- Run -------------------------------
if MODE == "save":
    save_checkpoint(CHECKPOINT_FILE)
    if DRIVE_BACKUP_DIR:
        backup_dir = Path(DRIVE_BACKUP_DIR)
        backup_dir.mkdir(parents=True, exist_ok=True)
        stamp = __import__("datetime").datetime.now().strftime("%Y%m%d_%H%M")
        shutil.copy(CHECKPOINT_FILE, backup_dir / f"vla_checkpoint_{stamp}.pt")
        print(f"Backed up to Drive: {backup_dir}")
    else:
        print("Note: /content is wiped when the session disconnects. Set")
        print("DRIVE_BACKUP_DIR, or files.download(str(CHECKPOINT_FILE)).")

elif MODE == "load":
    load_checkpoint(CHECKPOINT_FILE)
    print("-> you can now skip [4] and run [5] onwards")

elif MODE == "finetune":
    load_checkpoint(CHECKPOINT_FILE)
    print(f"Fine-tuning on {len(SAMPLES)} samples at lr={FINETUNE_LR}")
    finetune()
    save_checkpoint(CHECKPOINT_FILE)
    print("-> check the generated output in [5] onwards")

else:
    raise ValueError(f"MODE must be save / load / finetune, got: {MODE}")


In [ ]:
# =====================================================================
# [5] Generate: natural language -> Common Behavior Data
#     Autoregressive, with the predicted phase fed back each step, so the
#     learned state machine drives Idle -> Reach -> Grasp -> Carry -> Release.
# =====================================================================
# Instructions to generate from. Each produces one behavior dataset, written
# out in [6] in the same CBD-compatible format Demo A produces.
# Training captions are English, so keep the prompts English too.
PROMPTS = [
    "The person picks up the cup, drinks, and puts it back on the table.",
    "The person reaches for the cup on the table.",
    "The person carries the cup to another place and releases it.",
]
# To condition on an image as well, give a list of the same length
IMAGE_PATHS = None         # e.g. [str(CORPUS_DIR/"Episode1/.../frames/000030.jpg"), None, None]
PHASE_TEMPERATURE = 0.7    # 0 = deterministic (argmax); higher = more varied transitions
GENERATION_SEED = 0        # fixed seed for reproducibility (None = random each run)
GEN_SMOOTHING_ALPHA = 0.2  # post-generation temporal smoothing (0 disables it)
# True allows only the state transitions actually observed in training data:
# a hard grammar laid over the model's soft, learned transition probabilities.
ENFORCE_TRANSITION_GRAMMAR = True
MAX_GENERATION_FRAMES = MAX_SEQ_FRAMES


@torch.no_grad()
def generate(model, prompt=None, image_path=None,
             max_frames=MAX_GENERATION_FRAMES, temperature=PHASE_TEMPERATURE):
    assert prompt or image_path, "need at least one of prompt / image_path"
    model.eval()
    image = VisionEncoder.load_image(image_path) if image_path else None
    prefix = model.encode_prefix([prompt], [image], DEVICE)      # (1,2,D)
    tokens = torch.cat([prefix,
                        model.start_token.expand(1, 1, model.d_model)], dim=1)
    motions, phases = [], []
    for step in range(max_frames):
        seq = tokens + model.pos_encoding[:tokens.shape[1]]
        causal = nn.Transformer.generate_square_subsequent_mask(
            seq.shape[1], device=DEVICE)
        hidden = model.transformer(seq, mask=causal, is_causal=True)
        last = hidden[:, -1]
        motion_vec = model.motion_head(last)                      # (1,89)
        phase_logits = model.phase_head(last)                     # (1,P)
        if ENFORCE_TRANSITION_GRAMMAR:
            if phases:
                allowed = DATA_TRANSITIONS[PHASE_TO_ID[phases[-1]]] > 0
                allowed[PHASE_TO_ID[phases[-1]]] = True   # self-loops always allowed
            else:
                allowed = np.zeros(NUM_PHASES, dtype=bool)
                allowed[START_PHASE_IDS] = True           # start only where the data starts
            phase_logits = phase_logits.masked_fill(
                torch.tensor(~allowed, device=DEVICE).unsqueeze(0),
                float("-inf"))
        if temperature <= 0:
            phase_id = int(phase_logits.argmax(-1))
        else:
            probs = F.softmax(phase_logits / temperature, dim=-1)
            phase_id = int(torch.multinomial(probs, 1))
        if PHASE_VOCAB[phase_id] == "<end>" and step > 8:
            break
        # Re-normalise the quaternion block, same op as during training
        quats = F.normalize(motion_vec[:, QUAT_SLICE].view(1, -1, 4), dim=-1)
        motion_vec = torch.cat([quats.view(1, -1),
                                motion_vec[:, QUAT_SLICE.stop:]], dim=-1)
        motions.append(motion_vec[0].cpu().numpy())
        phases.append(PHASE_VOCAB[phase_id])
        next_token = model.motion_in(motion_vec) \
            + model.phase_embed(torch.tensor([phase_id], device=DEVICE))
        tokens = torch.cat([tokens, next_token.unsqueeze(1)], dim=1)
    motion = denormalize(np.stack(motions)) if motions else np.zeros((0, MOTION_DIM))
    return motion, phases


def postprocess_motion(motion):
    """Post-processing: make quaternion signs continuous, smooth over time,
    re-normalise. Removes the residual per-frame jitter of autoregressive
    generation."""
    if len(motion) < 2:
        return motion
    motion = motion.copy()
    quat_dim = len(BONE_ORDER) * 4
    quats = motion[:, :quat_dim].reshape(len(motion), -1, 4)
    for fi in range(1, len(motion)):        # q and -q are the same rotation; align signs
        flips = (quats[fi] * quats[fi - 1]).sum(-1) < 0
        quats[fi][flips] *= -1.0
    if GEN_SMOOTHING_ALPHA > 0:
        alpha = GEN_SMOOTHING_ALPHA
        for fi in range(1, len(motion)):
            motion[fi] = alpha * motion[fi] + (1 - alpha) * motion[fi - 1]
    quats = motion[:, :quat_dim].reshape(len(motion), -1, 4)
    norms = np.linalg.norm(quats, axis=-1, keepdims=True)
    motion[:, :quat_dim] = (quats / np.clip(norms, 1e-8, None)).reshape(
        len(motion), -1)
    if FREEZE_LOWER_BODY:
        # Frozen during training, but autoregressive drift can still nudge
        # the legs, so pin them once more after generation
        motion = freeze_lower_body(motion)
    return motion


# ---- Generate one dataset per instruction ----
GENERATIONS = []
image_paths = IMAGE_PATHS or [None] * len(PROMPTS)
for gi, (prompt, image_path) in enumerate(zip(PROMPTS, image_paths)):
    if GENERATION_SEED is not None:
        torch.manual_seed(GENERATION_SEED + gi)   # reproducible per instruction
    motion, phases = generate(MODEL, prompt, image_path)
    motion = postprocess_motion(motion)
    GENERATIONS.append({"prompt": prompt, "image_path": image_path,
                        "motion": motion, "phases": phases})
    print(f"\n[{gi + 1}/{len(PROMPTS)}] \"{prompt}\"")
    print(f"  {len(motion)} frames ({len(motion) / FPS:.1f}s)  phase sequence:")
    current = None
    for fi, phase in enumerate(phases + [None]):
        if phase != current:
            if current is not None:
                print(f"    {current}: through frame {fi - 1}")
            current = phase

# ---- How well was the state machine learned? Compare the transition matrix
#      measured from the data against the one the generations actually used. ----
gen_transitions = np.zeros((NUM_PHASES, NUM_PHASES))
for g in GENERATIONS:
    ids = [PHASE_TO_ID[p] for p in g["phases"]]
    for a, b in zip(ids, ids[1:]):
        gen_transitions[a, b] += 1
gen_row = gen_transitions.sum(axis=1, keepdims=True)
gen_transitions = np.divide(gen_transitions, gen_row,
                            out=np.zeros_like(gen_transitions),
                            where=gen_row > 0)
print("\nPhase transition matrix (top row: training data, bottom: generated)")
header = "            " + " ".join(f"{p[:7]:>8}" for p in PHASE_VOCAB)
print(header)
for i, phase in enumerate(PHASE_VOCAB):
    print(f"data {phase[:7]:>7} " + " ".join(
        f"{DATA_TRANSITIONS[i, j]:8.2f}" for j in range(NUM_PHASES)))
    print(f"gen  {phase[:7]:>7} " + " ".join(
        f"{gen_transitions[i, j]:8.2f}" for j in range(NUM_PHASES)))


In [ ]:
# =====================================================================
# [6] Export the generated datasets
#     One folder per instruction from [5], each containing a CBD-compatible
#     frames.jsonl, motion.vrma (Unity) and humanoid.xml + motion.npz +
#     replay_mujoco.py (MuJoCo).
#     This is the point of the whole demo: observed behavior and *generated*
#     behavior come out in the same schema and replay through the same
#     adapters. Nothing downstream can tell them apart.
# =====================================================================
import re as _re

GEN_ROOT = OUTPUT_ROOT / "generated_datasets"
if GEN_ROOT.exists():
    shutil.rmtree(GEN_ROOT)
GEN_ROOT.mkdir(parents=True)
NB = len(BONE_ORDER)

REPLAY_SOURCE = r"""'''Replay generated Common Behavior Data in the local MuJoCo viewer.

Usage:
    python replay_mujoco.py          # loop playback
    python replay_mujoco.py --once   # play once and hold the last frame

Keys (press inside the viewer window):
    SPACE : pause / resume
    R     : restart from frame 0
    , / . : step one frame backward / forward (while paused)
'''
import sys
import time
from pathlib import Path

import mujoco
import mujoco.viewer
import numpy as np

HERE = Path(__file__).parent
# Load the XML as a string. MjModel.from_xml_path opens the file down in C++
# and fails on non-ASCII paths, which is a common surprise on Windows.
model = mujoco.MjModel.from_xml_string(
    (HERE / "humanoid.xml").read_text(encoding="utf-8"))
data = mujoco.MjData(model)

archive = np.load(HERE / "motion.npz")
qpos = archive["qpos"]
fps = float(archive["fps"][0])
object_pos = archive["object_pos"] if "object_pos" in archive else None
total = len(qpos)
loop = "--once" not in sys.argv

print(f"frames={total}  fps={fps:.2f}  duration={total / fps:.2f}s  "
      f"loop={loop}")
print("keys: SPACE=pause/resume  R=restart  ,/.=step  (progress printed below)")

paused = False
frame = 0


def key_callback(keycode):
    '''Drive playback from the viewer keys (SPACE / R / , / .).'''
    global paused, frame
    key = chr(keycode) if 32 <= keycode < 127 else ""
    if keycode == 32:                     # SPACE
        paused = not paused
    elif key in ("r", "R"):
        frame = 0
    elif key == "." and paused:
        frame = min(frame + 1, total - 1)
    elif key == "," and paused:
        frame = max(frame - 1, 0)


with mujoco.viewer.launch_passive(model, data,
                                  key_callback=key_callback) as viewer:
    while viewer.is_running():
        step_start = time.time()
        index = min(frame, total - 1)
        data.qpos[:] = qpos[index]
        if object_pos is not None and model.nmocap > 0:
            data.mocap_pos[0] = object_pos[index]
        mujoco.mj_forward(model, data)
        viewer.sync()

        # Overwrite one progress line, so first/last frames are easy to spot
        marker = " <<< FIRST" if index == 0 else (
            " >>> LAST" if index == total - 1 else "")
        print(f"\rframe {index + 1:4d}/{total}  "
              f"t={index / fps:6.2f}s{'  [PAUSED]' if paused else ''}"
              f"{marker}          ", end="", flush=True)

        if not paused:
            if frame >= total - 1:
                if loop:
                    print("\n--- loop ---")
                    frame = 0
                else:
                    paused = True         # --once holds on the last frame
            else:
                frame += 1
        wait = 1.0 / fps - (time.time() - step_start)
        if wait > 0:
            time.sleep(wait)
print()
"""


def export_generation(prompt, image_path, motion, phases, GEN_DIR):
    GEN_DIR.mkdir(parents=True, exist_ok=True)
    PROMPT, IMAGE_PATH, GEN_PHASES = prompt, image_path, phases
    gen_quats_xyzw = motion[:, :NB * 4].reshape(-1, NB, 4)
    gen_hips = motion[:, NB * 4:NB * 4 + 3]
    gen_curls = {"Left": motion[:, NB * 4 + 3:NB * 4 + 8],
                 "Right": motion[:, NB * 4 + 8:NB * 4 + 13]}
    NUM_GEN = len(motion)
    times = np.arange(NUM_GEN, dtype=np.float32) / FPS

    # ---------------- (a) CBD-compatible frames.jsonl ----------------
    with open(GEN_DIR / "frames.jsonl", "w") as fp:
        for fi in range(NUM_GEN):
            fp.write(json.dumps({
                "frame": fi,
                "timestamp_sec": round(float(times[fi]), 4),
                "source": "vla_generated",
                "prompt": PROMPT, "image_prompt": IMAGE_PATH,
                "human": {
                    "bone_rotations_xyzw": {
                        bone: [round(float(v), 5) for v in gen_quats_xyzw[fi, bi]]
                        for bi, bone in enumerate(BONE_ORDER)},
                    "hips_position": [round(float(v), 4) for v in gen_hips[fi]],
                    "finger_curls_rad": {
                        side.lower(): [round(float(v), 3)
                                       for v in gen_curls[side][fi]]
                        for side in ["Left", "Right"]},
                },
                "phase": {"phase": GEN_PHASES[fi]},
            }) + "\n")

    # ---------------- (b) motion.vrma, identical format to Demo A cell [8] ----------------
    def q_mul(a, b):
        w1, x1, y1, z1 = a; w2, x2, y2, z2 = b
        return np.array([w1*w2 - x1*x2 - y1*y2 - z1*z2,
                         w1*x2 + x1*w2 + y1*z2 - z1*y2,
                         w1*y2 - x1*z2 + y1*w2 + z1*x2,
                         w1*z2 + x1*y2 - y1*x2 + z1*w2])

    VRM_NODE_DEFS = [
        ("hips", None, (0.0, 0.95, 0.0)), ("spine", "hips", (0.0, 0.10, 0.0)),
        ("chest", "spine", (0.0, 0.12, 0.0)), ("neck", "chest", (0.0, 0.20, 0.0)),
        ("head", "neck", (0.0, 0.06, 0.0)),
        ("leftUpperArm", "chest", (0.17, 0.14, 0.0)),
        ("leftLowerArm", "leftUpperArm", (0.26, 0.0, 0.0)),
        ("leftHand", "leftLowerArm", (0.25, 0.0, 0.0)),
        ("rightUpperArm", "chest", (-0.17, 0.14, 0.0)),
        ("rightLowerArm", "rightUpperArm", (-0.26, 0.0, 0.0)),
        ("rightHand", "rightLowerArm", (-0.25, 0.0, 0.0)),
        ("leftUpperLeg", "hips", (0.09, -0.05, 0.0)),
        ("leftLowerLeg", "leftUpperLeg", (0.0, -0.42, 0.0)),
        ("leftFoot", "leftLowerLeg", (0.0, -0.40, 0.0)),
        ("rightUpperLeg", "hips", (-0.09, -0.05, 0.0)),
        ("rightLowerLeg", "rightUpperLeg", (0.0, -0.42, 0.0)),
        ("rightFoot", "rightLowerLeg", (0.0, -0.40, 0.0)),
    ]
    COMMON_TO_VRM = {
        "hips": "hips", "spine": "spine", "chest": "chest", "neck": "neck",
        "head": "head", "left_upper_arm": "leftUpperArm",
        "left_lower_arm": "leftLowerArm", "left_hand": "leftHand",
        "right_upper_arm": "rightUpperArm", "right_lower_arm": "rightLowerArm",
        "right_hand": "rightHand", "left_upper_leg": "leftUpperLeg",
        "left_lower_leg": "leftLowerLeg", "left_foot": "leftFoot",
        "right_upper_leg": "rightUpperLeg", "right_lower_leg": "rightLowerLeg",
        "right_foot": "rightFoot"}
    FINGER_SEGMENTS = {"Thumb": ["Metacarpal", "Proximal", "Distal"],
                       "Index": ["Proximal", "Intermediate", "Distal"],
                       "Middle": ["Proximal", "Intermediate", "Distal"],
                       "Ring": ["Proximal", "Intermediate", "Distal"],
                       "Little": ["Proximal", "Intermediate", "Distal"]}
    FINGER_Z = {"Thumb": 0.030, "Index": 0.020, "Middle": 0.0,
                "Ring": -0.018, "Little": -0.034}
    for side, sign in [("left", 1.0), ("right", -1.0)]:
        for finger, segments in FINGER_SEGMENTS.items():
            parent = f"{side}Hand"
            for si, segment in enumerate(segments):
                translation = ((sign * 0.035, 0.0, FINGER_Z[finger]) if si == 0
                               else (sign * 0.030, 0.0, 0.0))
                VRM_NODE_DEFS.append((f"{side}{finger}{segment}", parent, translation))
    NODE_INDEX = {name: i for i, (name, _, _) in enumerate(VRM_NODE_DEFS)}

    def export_vrma(path):
        rotation_tracks = {}
        for bi, bone in enumerate(BONE_ORDER):
            vrm = COMMON_TO_VRM.get(bone)
            if vrm:
                rotation_tracks[vrm] = gen_quats_xyzw[:, bi].astype(np.float32)
        for side_key, side, axis in [("Left", "left", (0, 0, -1.0)),
                                     ("Right", "right", (0, 0, 1.0))]:
            for fi_i, finger in enumerate(["Thumb", "Index", "Middle", "Ring",
                                           "Little"]):
                per_seg = np.clip(gen_curls[side_key][:, fi_i] / 3.0, 0.0, 1.6)
                s = np.sin(per_seg / 2); c = np.cos(per_seg / 2)
                track = np.stack([axis[0]*s, axis[1]*s, axis[2]*s, c],
                                 axis=1).astype(np.float32)
                for segment in FINGER_SEGMENTS[finger]:
                    rotation_tracks[f"{side}{finger}{segment}"] = track
        hips_translation = np.stack(
            [gen_hips[:, 0], 0.95 + gen_hips[:, 1], gen_hips[:, 2]],
            axis=1).astype(np.float32)

        blob, views, accessors = bytearray(), [], []
        def add(array, atype, minmax=False):
            data = array.astype(np.float32).tobytes()
            offset = len(blob); blob.extend(data)
            while len(blob) % 4: blob.append(0)
            views.append({"buffer": 0, "byteOffset": offset, "byteLength": len(data)})
            acc = {"bufferView": len(views) - 1, "componentType": 5126,
                   "count": len(array), "type": atype}
            if minmax:
                flat = array.reshape(len(array), -1)
                acc["min"] = [float(v) for v in flat.min(0)]
                acc["max"] = [float(v) for v in flat.max(0)]
            accessors.append(acc); return len(accessors) - 1

        t_acc = add(times.reshape(-1, 1), "SCALAR", True)
        samplers, channels = [], []
        for name, track in rotation_tracks.items():
            samplers.append({"input": t_acc, "output": add(track, "VEC4"),
                             "interpolation": "LINEAR"})
            channels.append({"sampler": len(samplers) - 1,
                             "target": {"node": NODE_INDEX[name], "path": "rotation"}})
        samplers.append({"input": t_acc, "output": add(hips_translation, "VEC3"),
                         "interpolation": "LINEAR"})
        channels.append({"sampler": len(samplers) - 1,
                         "target": {"node": NODE_INDEX["hips"],
                                    "path": "translation"}})
        nodes = [{"name": n, "translation": list(t)} for n, _, t in VRM_NODE_DEFS]
        for i, (name, parent, _) in enumerate(VRM_NODE_DEFS):
            if parent:
                nodes[NODE_INDEX[parent]].setdefault("children", []).append(i)
        human_bones = {n: {"node": NODE_INDEX[n]} for n in NODE_INDEX}
        gltf = {"asset": {"version": "2.0", "generator": "vla_generated"},
                "extensionsUsed": ["VRMC_vrm_animation"],
                "extensions": {"VRMC_vrm_animation": {
                    "specVersion": "1.0",
                    "humanoid": {"humanBones": human_bones}}},
                "scene": 0, "scenes": [{"nodes": [NODE_INDEX["hips"]]}],
                "nodes": nodes,
                "animations": [{"name": "generated", "samplers": samplers,
                                "channels": channels}],
                "buffers": [{"byteLength": len(blob)}],
                "bufferViews": views, "accessors": accessors}
        jb = json.dumps(gltf, separators=(",", ":")).encode()
        while len(jb) % 4: jb += b" "
        bb = bytes(blob)
        with open(path, "wb") as fp:
            fp.write(struct.pack("<III", 0x46546C67, 2, 12 + 8 + len(jb) + 8 + len(bb)))
            fp.write(struct.pack("<II", len(jb), 0x4E4F534A)); fp.write(jb)
            fp.write(struct.pack("<II", len(bb), 0x004E4942)); fp.write(bb)

    export_vrma(GEN_DIR / "motion.vrma")

    # ---------------- (c) MuJoCo（humanoid.xml + motion.npz + replay） ----------------
    import mujoco

    FINGER_ORDER = ["thumb", "index", "middle", "ring", "little"]
    FINGER_Y = {"thumb": -0.042, "index": -0.024, "middle": -0.008,
                "ring": 0.008, "little": 0.024}
    def _fingers(side):
        prefix, sign = side.lower(), (1.0 if side == "Left" else -1.0)
        axis = "0 1 0" if side == "Left" else "0 -1 0"
        return "".join(
            f'<body name="{prefix}_{f}" pos="{sign*0.088:.3f} {FINGER_Y[f]:.3f} 0">'
            f'<joint name="{prefix}_finger_{f}" type="hinge" axis="{axis}" '
            f'range="0 2.2" limited="true"/>'
            f'<geom type="capsule" fromto="0 0 0 {sign*0.055:.3f} 0 0" size="0.008" '
            f'rgba="0.8 0.8 0.85 1"/></body>' for f in FINGER_ORDER)

    RGBA = 'rgba="0.55 0.62 0.72 1"'
    ACC = 'rgba="0.85 0.45 0.25 1"'
    HUMANOID_XML = f"""<mujoco model="vla_generated">
      <option gravity="0 0 0"/>
      <visual><quality offsamples="0"/>
        <headlight ambient="0.45 0.45 0.45" diffuse="0.7 0.7 0.7"/></visual>
      <asset><texture name="grid" type="2d" builtin="checker" width="256"
          height="256" rgb1="0.20 0.24 0.30" rgb2="0.26 0.30 0.36"/>
        <material name="grid" texture="grid" texrepeat="6 6"/></asset>
      <worldbody>
        <geom name="floor" type="plane" size="4 4 0.05" material="grid"/>
        <light pos="0 -2 3" dir="0 0.5 -1"/>
        <body name="pelvis" pos="0 0 0.93"><freejoint name="root"/>
          <geom type="box" size="0.11 0.08 0.07" {RGBA}/>
          <body name="spine" pos="0 0 0.10"><joint name="spine" type="ball"/>
            <geom type="capsule" fromto="0 0 0 0 0 0.10" size="0.07" {RGBA}/>
            <body name="chest" pos="0 0 0.12"><joint name="chest" type="ball"/>
              <geom type="capsule" fromto="0 0 0 0 0 0.14" size="0.09" {RGBA}/>
              <body name="neck" pos="0 0 0.20"><joint name="neck" type="ball"/>
                <geom type="capsule" fromto="0 0 0 0 0 0.05" size="0.035" {RGBA}/>
                <body name="head" pos="0 0 0.06"><joint name="head" type="ball"/>
                  <geom type="sphere" pos="0 0 0.08" size="0.09" {RGBA}/></body></body>
              <body name="left_shoulder" pos="0.08 0 0.14">
                <geom type="sphere" size="0.045" {RGBA}/>
                <body name="left_upper_arm" pos="0.09 0 0">
                  <joint name="left_upper_arm" type="ball"/>
                  <geom type="capsule" fromto="0 0 0 0.26 0 0" size="0.038" {RGBA}/>
                  <body name="left_lower_arm" pos="0.26 0 0">
                    <joint name="left_lower_arm" type="ball"/>
                    <geom type="capsule" fromto="0 0 0 0.25 0 0" size="0.032" {RGBA}/>
                    <body name="left_hand" pos="0.25 0 0">
                      <joint name="left_hand" type="ball"/>
                      <geom type="box" pos="0.045 0 0" size="0.045 0.032 0.012" {ACC}/>
                      {_fingers('Left')}</body></body></body></body>
              <body name="right_shoulder" pos="-0.08 0 0.14">
                <geom type="sphere" size="0.045" {RGBA}/>
                <body name="right_upper_arm" pos="-0.09 0 0">
                  <joint name="right_upper_arm" type="ball"/>
                  <geom type="capsule" fromto="0 0 0 -0.26 0 0" size="0.038" {RGBA}/>
                  <body name="right_lower_arm" pos="-0.26 0 0">
                    <joint name="right_lower_arm" type="ball"/>
                    <geom type="capsule" fromto="0 0 0 -0.25 0 0" size="0.032" {RGBA}/>
                    <body name="right_hand" pos="-0.25 0 0">
                      <joint name="right_hand" type="ball"/>
                      <geom type="box" pos="-0.045 0 0" size="0.045 0.032 0.012" {ACC}/>
                      {_fingers('Right')}</body></body></body></body></body></body>
          <body name="left_upper_leg" pos="0.09 0 -0.05">
            <joint name="left_upper_leg" type="ball"/>
            <geom type="capsule" fromto="0 0 0 0 0 -0.42" size="0.055" {RGBA}/>
            <body name="left_lower_leg" pos="0 0 -0.42">
              <joint name="left_lower_leg" type="ball"/>
              <geom type="capsule" fromto="0 0 0 0 0 -0.40" size="0.045" {RGBA}/>
              <body name="left_foot" pos="0 0 -0.40">
                <joint name="left_foot" type="ball"/>
                <geom type="box" pos="0 -0.05 -0.025" size="0.045 0.10 0.02" {ACC}/>
              </body></body></body>
          <body name="right_upper_leg" pos="-0.09 0 -0.05">
            <joint name="right_upper_leg" type="ball"/>
            <geom type="capsule" fromto="0 0 0 0 0 -0.42" size="0.055" {RGBA}/>
            <body name="right_lower_leg" pos="0 0 -0.42">
              <joint name="right_lower_leg" type="ball"/>
              <geom type="capsule" fromto="0 0 0 0 0 -0.40" size="0.045" {RGBA}/>
              <body name="right_foot" pos="0 0 -0.40">
                <joint name="right_foot" type="ball"/>
                <geom type="box" pos="0 -0.05 -0.025" size="0.045 0.10 0.02" {ACC}/>
              </body></body></body>
        </body>
      </worldbody>
    </mujoco>"""
    (GEN_DIR / "humanoid.xml").write_text(HUMANOID_XML, encoding="utf-8")
    mj_model = mujoco.MjModel.from_xml_path(str(GEN_DIR / "humanoid.xml"))

    Q_C2MJ = np.array([math.cos(math.pi/4), math.sin(math.pi/4), 0.0, 0.0])
    def to_mj_quat(q_xyzw):
        q = np.array([q_xyzw[3], q_xyzw[0], q_xyzw[1], q_xyzw[2]])   # wxyz
        r = q_mul(Q_C2MJ, q_mul(q, Q_C2MJ * [1, -1, -1, -1]))
        return r / np.linalg.norm(r)

    QPOS = np.zeros((NUM_GEN, mj_model.nq), np.float32)
    bone_index = {b: i for i, b in enumerate(BONE_ORDER)}
    for joint_id in range(mj_model.njnt):
        name = mujoco.mj_id2name(mj_model, mujoco.mjtObj.mjOBJ_JOINT, joint_id)
        adr = int(mj_model.jnt_qposadr[joint_id])
        jtype = int(mj_model.jnt_type[joint_id])
        for fi in range(NUM_GEN):
            if jtype == int(mujoco.mjtJoint.mjJNT_FREE):
                h = gen_hips[fi]
                QPOS[fi, adr:adr+3] = [h[0], -h[2], h[1] + 0.93]
                QPOS[fi, adr+3:adr+7] = to_mj_quat(gen_quats_xyzw[fi, bone_index["hips"]])
            elif jtype == int(mujoco.mjtJoint.mjJNT_BALL):
                QPOS[fi, adr:adr+4] = to_mj_quat(gen_quats_xyzw[fi, bone_index[name]])
            else:
                side = "Left" if name.startswith("left") else "Right"
                finger = name.split("_")[-1]
                QPOS[fi, adr] = float(np.clip(
                    gen_curls[side][fi, FINGER_ORDER.index(finger)], 0.0, 2.2))

    np.savez_compressed(GEN_DIR / "motion.npz", qpos=QPOS,
                        fps=np.array([FPS], np.float32),
                        timestamp_sec=times)
    shutil.copy
    (GEN_DIR / "replay_mujoco.py").write_text(REPLAY_SOURCE,
                                              encoding="utf-8")


# ---- Write one dataset per instruction ----
MANIFEST = []
for gi, g in enumerate(GENERATIONS):
    slug = _re.sub(r"[^a-z0-9]+", "_", g["prompt"].lower())[:40].strip("_") \
        or f"prompt_{gi + 1}"
    out_dir = GEN_ROOT / f"{gi + 1:02d}_{slug}"
    export_generation(g["prompt"], g["image_path"], g["motion"], g["phases"],
                      out_dir)
    MANIFEST.append({"index": gi + 1, "prompt": g["prompt"],
                     "image_prompt": g["image_path"],
                     "frames": len(g["motion"]),
                     "duration_sec": round(len(g["motion"]) / FPS, 2),
                     "folder": out_dir.name})
    print(f"[{gi + 1}/{len(GENERATIONS)}] {out_dir.name}  "
          f"({len(g['motion'])} frames)")

(GEN_ROOT / "manifest.json").write_text(json.dumps({
    "note": ("Each folder is one behavior dataset generated from the "
             "natural-language instruction in 'prompt'. Same schema and "
             "adapters as observed Common Behavior Data."),
    "generation_seed": GENERATION_SEED,
    "smoothing_alpha": GEN_SMOOTHING_ALPHA,
    "lower_body_frozen": bool(FREEZE_LOWER_BODY),
    "lower_body_pose": FREEZE_POSE if FREEZE_LOWER_BODY else None,
    "lower_body_note": ("Legs are pinned to the T-pose because the current "
                        "corpus is seated and small; revisit when more "
                        "episodes with visible lower body are available."),
    "generations": MANIFEST}, indent=2, ensure_ascii=False), encoding="utf-8")
print("\nGenerated datasets:", GEN_ROOT)


In [ ]:
# =====================================================================
# [7] Zip the generated datasets and download them
#     Bundles only generated_datasets/ -- the per-instruction CBD-compatible
#     datasets plus manifest.json.
#     Training artifacts such as the checkpoint stay in _vla_runs and are
#     deliberately left out of the ZIP.
# =====================================================================
import shutil
from google.colab import files

GEN_ROOT_DIR = "/content/behavior_corpus/_vla_runs/generated_datasets"
ZIP_PATH = "/content/vla_generated_datasets"

shutil.make_archive(
    ZIP_PATH,
    "zip",
    root_dir="/content/behavior_corpus/_vla_runs",
    base_dir="generated_datasets",
)
files.download(ZIP_PATH + ".zip")


## Where this goes next

| Step | What it means |
|---|---|
| More data | Add episode folders, re-run `[1.6]` → `[4]`. Nothing else changes. |
| Unfreeze the lower body | Add standing / walking episodes, set `FREEZE_LOWER_BODY = False` |
| Include objects | Also generate the `objects` proxy positions in `frames.jsonl`, for behavior with objects |
| Tokenise motion | Discretise motion with a VQ-VAE and treat it as next-token prediction, like a language model |
| Diffusion | Replace the motion head with denoising diffusion for more varied generation |
| Real vision | Fine-tune the frozen ResNet instead of freezing it (A100 recommended) |

### Try the output

The generated `motion.vrma` plays in Unity's SimpleVrma scene, and
`humanoid.xml` + `motion.npz` + `replay_mujoco.py` plays in a local MuJoCo
viewer:

```bash
pip install mujoco
cd 01_the_person_picks_up_the_cup_...
python replay_mujoco.py          # loop
python replay_mujoco.py --once   # play once, hold the final pose
```

**Observed behavior and generated behavior replay through the same adapters,
in the same schema.** That property — not the size of this model — is what the
demo is meant to show.

### Tuning generation

| Symptom | Fix |
|---|---|
| Motion is jittery | Lower `GEN_SMOOTHING_ALPHA` (default 0.2). Training already applies velocity/acceleration losses and scheduled sampling. |
| Results differ every run | Set `GENERATION_SEED` to a fixed value (default 0); `None` randomises. |
| Output too short / too long | Lower `PHASE_TEMPERATURE` (0.7 → 0.3) to be more decisive; `MAX_GENERATION_FRAMES` caps the length. |
| Impossible phase transitions | Keep `ENFORCE_TRANSITION_GRAMMAR = True` (default). |
| Every instruction gives the same motion | Not enough episodes — add episodes of *different* behaviors. |
| Loss does not converge | Increase `EPOCHS`. A quaternion loss below 0.01 is well converged (0.0036 ≈ 5 degrees). |
